# Marketing Campaign Analyzer - AI Prompt Evaluation

This notebook evaluates the three AI functions in `services/ai_service.py`
using **real parsed campaign data** from the CSV exports.


In [ ]:
import os, json, re, time
from pathlib import Path
from openai import AsyncOpenAI
from pydantic import BaseModel, ValidationError
from typing import Literal
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
client = AsyncOpenAI()
DATA_DIR = Path().resolve().parent / 'data'

key_ok = bool(os.getenv('OPENAI_API_KEY'))
print(f'OpenAI key: {"present" if key_ok else "MISSING - set OPENAI_API_KEY in .env"}')
print(f'data/meta_export.csv: {(DATA_DIR/"meta_export.csv").exists()}')
print(f'data/google_export.csv: {(DATA_DIR/"google_export.csv").exists()}')


OpenAI key: present
data/meta_export.csv: True
data/google_export.csv: True


---
## 1. Re-parse real CSV files → build AI payloads

We import `parse_csv()` output from Notebook 1.
This is the same dict that `routers/analyze.py` passes to `ai_service.py`.


In [ ]:
META_COLUMNS = {
    "campaign_name": ["Campaign name","Campaign Name"],
    "impressions": ["Impressions"],
    "clicks": ["Clicks (all)","Clicks"],
    "spend": ["Amount spent (EUR)","Amount spent (USD)","Amount spent"],
    "conversions": ["Results","Conversions"],
    "revenue": ["Purchase conversion value","Conv. value"],
}
GOOGLE_COLUMNS = {
    "campaign_name": ["Campaign"],
    "impressions": ["Impr.","Impressions"],
    "clicks": ["Clicks"],
    "spend": ["Cost","Spend"],
    "conversions": ["Conversions"],
    "revenue": ["Conv. value","Conversion value"],
}

def _detect(df):
    c = set(df.columns.str.lower())
    if any("amount spent" in x for x in c): return "meta"
    if any("avg. cpc" in x for x in c) or "impr." in c: return "google"
    return "unknown"
def _find(df, cands):
    for c in cands:
        if c in df.columns: return c
    lm = {col.lower(): col for col in df.columns}
    for c in cands:
        if c.lower() in lm: return lm[c.lower()]
    return None
def _sf(val):
    if pd.isna(val): return 0.0
    s = str(val).strip()
    if not s or s=="--": return 0.0
    s = re.sub(r'[^\d,.\-]','',s)
    if ',' in s and '.' in s and s.rfind(',')>s.rfind('.'):
        s=s.replace('.','').replace(',','.')
    else: s=s.replace(',','')
    try: return float(s)
    except: return 0.0

def build_payload(path: Path, avg_order_value: float = None) -> tuple:
    """Returns (platform, campaigns_list, metrics_json_string) — mirrors parse_csv()."""
    df = pd.read_csv(path).dropna(how='all')
    plat = _detect(df)
    cm = META_COLUMNS if plat == 'meta' else GOOGLE_COLUMNS
    name_c = _find(df, cm['campaign_name'])
    imp_c = _find(df, cm['impressions'])
    clk_c = _find(df, cm['clicks'])
    spd_c = _find(df, cm['spend'])
    conv_c = _find(df, cm['conversions'])
    rev_c = _find(df, cm.get('revenue', []))
    records = []
    for _, row in df.iterrows():
        name = str(row.get(name_c,'')).strip()
        if not name: continue
        imp = int(_sf(row.get(imp_c,0)))
        clk = int(_sf(row.get(clk_c,0)))
        spd = _sf(row.get(spd_c,0))
        conv = _sf(row.get(conv_c,0)) if conv_c else 0.0
        rev = _sf(row.get(rev_c, 0)) if rev_c  else 0.0
        ctr = (clk/imp*100) if imp>0 else 0.0
        cpc = (spd/clk) if clk>0 else 0.0
        cpa = (spd/conv) if conv>0 else 0.0
        if rev>0: roas=round(rev/spd,2)
        elif avg_order_value and conv>0: roas=round((conv*avg_order_value)/spd,2)
        else: roas=0.0
        records.append({'campaign_name':name,'impressions':imp,'clicks':clk,
                        'spend':round(spd,2),'conversions':round(conv,2),
                        'ctr':round(ctr,2),'cpc':round(cpc,2),
                        'cpa':round(cpa,2),'roas':roas})
    return plat, records, json.dumps(records, indent=2)

meta_platform, meta_campaigns, meta_json = build_payload(DATA_DIR/'meta_export.csv', avg_order_value=80)
google_platform, google_campaigns, google_json = build_payload(DATA_DIR/'google_export.csv')

print(f'Meta payload : {len(meta_campaigns)} campaigns  ({len(meta_json):,} chars)')
print(f'Google payload : {len(google_campaigns)} campaigns ({len(google_json):,} chars)')
print()
print('First record going to the AI (Meta):')
print(json.dumps(meta_campaigns[0], indent=2))


Meta payload : 10 campaigns  (2,235 chars)
Google payload : 10 campaigns (2,174 chars)

First record going to the AI (Meta):
{
  "campaign_name": "Black Friday - Prospecting",
  "impressions": 312400,
  "clicks": 6820,
  "spend": 9840.5,
  "conversions": 142.0,
  "ctr": 2.18,
  "cpc": 1.44,
  "cpa": 69.3,
  "roas": 1.15
}


---
## 2. Pydantic schemas - mirrors `models/schemas.py`

Every AI response must pass validation against these models before `routers/analyze.py`
returns a `200` to the frontend. If validation fails here, it will fail in production.


In [ ]:
# Mirrors models/schemas.py
class Recommendation(BaseModel):
    title: str
    description: str
    impact_score: int
    category: Literal['budget','audience','creative','bidding','targeting']
    action: str

class BudgetSuggestion(BaseModel):
    campaign_name: str
    current_budget: float
    suggested_budget: float
    reason: str

class CopyVariant(BaseModel):
    text: str
    score: float
    score_breakdown: dict
    reason: str

class TopCombination(BaseModel):
    headline: str
    primary_text: str
    cta: str
    total_score: float
    predicted_ctr: str

print('Pydantic models loaded - matching models/schemas.py')


Pydantic models loaded - matching models/schemas.py


---
## 3. `analyse_campaigns()` - Meta data

This is `ANALYSIS_SYSTEM` + real Meta payload sent to gpt-4o-mini,
exactly as it happens inside `services/ai_service.py`.


In [ ]:
# Mirrors ANALYSIS_SYSTEM in ai_service.py
ANALYSIS_SYSTEM = (
    'You are a senior performance marketing analyst with 10+ years experience '
    'in Meta Ads and Google Ads. Analyse campaign data and provide actionable recommendations.\n\n'
    'Respond ONLY with valid JSON matching this exact schema:\n'
    '{\n'
    '  "recommendations": [\n'
    '    {"title": str, "description": "2-3 sentences — must include campaign names and exact numbers",\n'
    '     "impact_score": int 1-10, "category": "budget|audience|creative|bidding|targeting",\n'
    '     "action": "starts with a verb — one concrete next step"}\n'
    '  ],\n'
    '  "budget_suggestions": [\n'
    '    {"campaign_name": str, "current_budget": float, "suggested_budget": float, "reason": str}\n'
    '  ],\n'
    '  "summary_insight": "2-3 sentence executive summary of account health"\n'
    '}\n'
    'Rank recommendations by impact_score descending. Maximum 5 recommendations.'
)

async def call_llm(system, user, model='gpt-4o-mini', temperature=0.4, json_mode=True):
    """Mirrors _chat() in ai_service.py."""
    kw = {'response_format': {'type': 'json_object'}} if json_mode else {}
    start = time.time()
    try:
        resp = await client.chat.completions.create(
            model=model, temperature=temperature, max_tokens=4000,
            messages=[{'role':'system','content':system},{'role':'user','content':user}],
            **kw)
        raw = resp.choices[0].message.content
        data = json.loads(raw) if json_mode else raw
        return {'data': data, 'latency_ms': int((time.time()-start)*1000),
                'tokens': resp.usage.total_tokens, 'error': None}
    except Exception as e:
        return {'data': None, 'latency_ms': int((time.time()-start)*1000),
                'tokens': 0, 'error': str(e)}

meta_user_prompt = (
    f'Platform: {meta_platform}  |  Currency: EUR\n\n'
    f'Campaign data ({len(meta_campaigns)} campaigns):\n{meta_json}\n\n'
    'Analyse spend efficiency, identify poor ROAS/high CPA/zero-conversion campaigns, '
    'and give 5 ranked recommendations. Reference campaign names and exact numbers.'
)

print(f'Sending {len(meta_campaigns)} Meta campaigns to gpt-4o-mini...')
meta_result = await call_llm(ANALYSIS_SYSTEM, meta_user_prompt, temperature=0.3)
print(f'  Result: {meta_result}')
print(f'  Latency : {meta_result["latency_ms"]} ms')
print(f'  Tokens  : {meta_result["tokens"]}')
print(f'  Error   : {meta_result["error"] or "none"}')


Sending 10 Meta campaigns to gpt-4o-mini...
  Result: {'data': {'recommendations': [{'title': 'Reallocate Budget from Poor Performing Campaigns', 'description': "The 'Spring Collection - Lookalike 3%' campaign has a low ROAS of 0.76 and a high CPA of 104.74. Consider reallocating its budget to more efficient campaigns like 'Dynamic Product Ads - Cart Abandoners' which has a ROAS of 4.94.", 'impact_score': 10, 'category': 'budget', 'action': "Reallocate budget from 'Spring Collection - Lookalike 3%' to 'Dynamic Product Ads - Cart Abandoners'."}, {'title': 'Pause Underperforming Brand Awareness Campaign', 'description': "The 'Brand Awareness - Video' campaign has zero conversions and a ROAS of 0.0 after spending 2980.0 EUR. Pausing this campaign could free up budget for more effective campaigns.", 'impact_score': 9, 'category': 'budget', 'action': "Pause the 'Brand Awareness - Video' campaign."}, {'title': 'Optimize Targeting for Competitor Conquest Campaign', 'description': "The 'Compet

### 3a. Schema validation against `models/schemas.py`


In [ ]:
data = meta_result['data']
recs = data.get('recommendations', []) if data else []
buds = data.get('budget_suggestions', []) if data else []

print('Schema validation (Recommendation model):')
schema_ok = True
for i, r in enumerate(recs):
    try:
        Recommendation(**r)
        print(f'  rec #{i+1} ✅')
    except ValidationError as e:
        schema_ok = False
        print(f'  rec #{i+1} ❌  {e.errors()[0]["loc"]} — {e.errors()[0]["msg"]}')

print()
print('Schema validation (BudgetSuggestion model):')
for i, b in enumerate(buds):
    try:
        BudgetSuggestion(**b)
        print(f'  suggestion #{i+1} ✅')
    except ValidationError as e:
        schema_ok = False
        print(f'  suggestion #{i+1} ❌  {e.errors()[0]["msg"]}')

print()
print(f'Overall schema: {"✅ all valid — would pass routers/analyze.py" if schema_ok else "❌ FAILS — fix schema or prompt"}')


Schema validation (Recommendation model):
  rec #1 ✅
  rec #2 ✅
  rec #3 ✅
  rec #4 ✅
  rec #5 ✅

Schema validation (BudgetSuggestion model):
  suggestion #1 ✅
  suggestion #2 ✅
  suggestion #3 ✅
  suggestion #4 ✅

Overall schema: ✅ all valid — would pass routers/analyze.py


### 3b. Content quality - are recommendations specific to our real data?


In [ ]:
real_campaign_names = [c['campaign_name'] for c in meta_campaigns]
real_numbers = set()
for c in meta_campaigns:
    real_numbers.update([str(round(c['spend'])), str(round(c['cpa'])), str(c['impressions'])])

print(f'Recommendations from the AI ({len(recs)} total):')
print('=' * 72)
action_verbs = {'increase','decrease','pause','reduce','test','switch',
                'reallocate','raise','lower','scale','launch','move',
                'shift','expand','cut','set','consolidate','duplicate'}

for i, rec in enumerate(recs, 1):
    body = rec.get('description','') + ' ' + rec.get('action','')
    mentioned = [n for n in real_campaign_names if n[:20] in body]
    has_numbers = bool(re.search(r'\d+\.?\d*', body))
    first_word = rec.get('action','').split()[0].lower() if rec.get('action') else ''
    action_ok = first_word in action_verbs

    print(f'\n#{i}  [{rec.get("category","?").upper()}]  impact={rec.get("impact_score")}/10')
    print(f'Title: {rec.get("title","")}')
    print(f'Desc : {rec.get("description","")[:130]}')
    print(f'Action : {rec.get("action","")}')
    print(f'Quality: '
          f'mentions_campaign={"✅ " + str(mentioned) if mentioned else "⚠️  none (generic)"}  '
          f'has_numbers={"✅" if has_numbers else "⚠️"}  '
          f'verb={"✅" if action_ok else "⚠️ check"}')

print('\n' + '=' * 72)
print('Executive summary:')
print(data.get('summary_insight',''))


Recommendations from the AI (5 total):

#1  [BUDGET]  impact=10/10
Title: Reallocate Budget from Poor Performing Campaigns
Desc : The 'Spring Collection - Lookalike 3%' campaign has a low ROAS of 0.76 and a high CPA of 104.74. Consider reallocating its budget 
Action : Reallocate budget from 'Spring Collection - Lookalike 3%' to 'Dynamic Product Ads - Cart Abandoners'.
Quality: mentions_campaign=✅ ['Spring Collection - Lookalike 3%', 'Spring Collection - Broad EU', 'Dynamic Product Ads - Cart Abandoners']  has_numbers=✅  verb=✅

#2  [BUDGET]  impact=9/10
Title: Pause Underperforming Brand Awareness Campaign
Desc : The 'Brand Awareness - Video' campaign has zero conversions and a ROAS of 0.0 after spending 2980.0 EUR. Pausing this campaign cou
Action : Pause the 'Brand Awareness - Video' campaign.
Quality: mentions_campaign=✅ ['Brand Awareness - Video']  has_numbers=✅  verb=✅

#3  [TARGETING]  impact=8/10
Title: Optimize Targeting for Competitor Conquest Campaign
Desc : The 'Competitor 

---
## 4. `analyse_campaigns()` - Google data

Google has real ROAS from `Conv. value` - we check if the AI correctly
identifies the negative-ROAS campaigns that are draining budget.


In [ ]:
google_user_prompt = (
    f'Platform: {google_platform}  |  Currency: EUR\n\n'
    f'Campaign data ({len(google_campaigns)} campaigns, ROAS from actual Conv. value):\n'
    f'{google_json}\n\n'
    'Identify the highest-ROI campaigns vs budget drains. '
    'Give 5 ranked recommendations. Reference exact campaign names, spend and ROAS.'
)

print(f'Sending {len(google_campaigns)} Google campaigns to gpt-4o-mini...')
google_result = await call_llm(ANALYSIS_SYSTEM, google_user_prompt, temperature=0.3)
print(f'Latency : {google_result["latency_ms"]} ms')
print(f'Tokens : {google_result["tokens"]}')

g_data = google_result['data']
g_recs = g_data.get('recommendations', []) if g_data else []
g_names = [c['campaign_name'] for c in google_campaigns]

low_roas = [c['campaign_name'] for c in google_campaigns if 0 < c['roas'] < 1.5]
print(f'\nCampaigns with ROAS < 1.5x (AI should flag these): {low_roas}')
print()

print(f'Recommendations ({len(g_recs)} total):')
print('=' * 72)
for i, rec in enumerate(g_recs, 1):
    body = rec.get('description','') + ' ' + rec.get('action','')
    mentioned = [n for n in g_names if n[:20] in body]
    flags_low = any(n[:15] in body for n in low_roas)
    has_numbers = bool(re.search(r'\d+\.?\d*', body))
    print(f'\n#{i}  [{rec.get("category","?").upper()}]  impact={rec.get("impact_score")}/10')
    print(f'Title : {rec.get("title","")}')
    print(f'Desc : {rec.get("description","")[:130]}')
    print(f'Action : {rec.get("action","")}')
    print(f'Quality: mentions_campaign={"✅" if mentioned else "⚠️"}  '
          f'flags_low_roas={"✅" if flags_low else "⚠️"}  '
          f'has_numbers={"✅" if has_numbers else "⚠️"}')

print('\n' + '=' * 72)
print('Executive summary:')
print(g_data.get('summary_insight','') if g_data else 'N/A')


Sending 10 Google campaigns to gpt-4o-mini...
Latency : 9961 ms
Tokens : 1799

Campaigns with ROAS < 1.5x (AI should flag these): ['Generic - Running Shoes', 'Generic - Sports Apparel', 'Competitor - Nike', 'Competitor - Adidas', 'Display - Remarketing', 'YouTube - Prospecting']

Recommendations (5 total):

#1  [BUDGET]  impact=9/10
Title : Increase Budget for Brand Search - Exact
Desc : The Brand Search - Exact campaign has a high ROAS of 8.0 with a spend of €3120. Increasing the budget could further capitalize on 
Action : Increase the budget by 20% to maximize returns
Quality: mentions_campaign=✅  flags_low_roas=⚠️  has_numbers=✅

#2  [BUDGET]  impact=8/10
Title : Pause Generic - Running Shoes
Desc : The Generic - Running Shoes campaign has a low ROAS of 0.78 and a high spend of €8940. Pausing this campaign will help reallocate 
Action : Pause the campaign to stop further losses
Quality: mentions_campaign=✅  flags_low_roas=✅  has_numbers=✅

#3  [CREATIVE]  impact=7/10
Title : Optimi

---
## 5. `generate_copy()` - `POST /api/generate-copy`

We use the Meta campaign context (product: running shoes, audience inferred from
retargeting campaigns) to test the copy generator with a realistic scenario.


In [ ]:
# Mirrors COPY_SYSTEM in ai_service.py
COPY_SYSTEM = (
    'You are a world-class direct response copywriter for digital advertising.\n\n'
    'Respond ONLY with valid JSON matching this schema:\n'
    '{\n'
    '  "headlines": [{"text": str, "score": 0-10,\n'
    '    "score_breakdown": {"relevance":0-10,"emotion":0-10,"clarity":0-10,"platform_fit":0-10},\n'
    '    "reason": "1 sentence"}],\n'
    '  "primary_texts": [same structure],\n'
    '  "ctas": [same structure, 2-5 words only],\n'
    '  "top_combinations": [{"headline":str,"primary_text":str,"cta":str,\n'
    '    "total_score":0-10,"predicted_ctr":"e.g. 2.1%-3.4%"}]\n'
    '}\n'
    'Generate exactly: 5 headlines, 5 primary_texts, 3 ctas, 3 top_combinations.'
)

copy_user = (
    'Platform: meta | Objective: conversions | Tone: urgent | Language: english\n\n'
    'Target audience: Online shoppers who visited the product page but did not purchase '
    '(matched to our Dynamic Product Ads - Cart Abandoners campaign audience)\n\n'
    'Product/Service: Premium running shoes - 30% seasonal discount, '
    'free shipping on orders over €50, limited stock\n\n'
    'Context: This copy will run alongside our best-performing campaign '
    '(Dynamic Product Ads - Cart Abandoners, CTR 8.80%, CPA €16.19). '
    'Match the urgency that makes that audience convert.'
)

print('Sending copy generation request to gpt-4o...')
copy_result = await call_llm(COPY_SYSTEM, copy_user, model='gpt-4o', temperature=0.75)
print(f'Latency : {copy_result["latency_ms"]} ms')
print(f'Tokens : {copy_result["tokens"]}')


Sending copy generation request to gpt-4o...
Latency : 8162 ms
Tokens : 1620


### 5a. Schema validation + count checks


In [ ]:
copy_data = copy_result['data']
headlines = copy_data.get('headlines', []) if copy_data else []
pri_texts = copy_data.get('primary_texts', []) if copy_data else []
ctas = copy_data.get('ctas', []) if copy_data else []
combos = copy_data.get('top_combinations', []) if copy_data else []

print('Count checks (must match CopyResponse schema):')
for label, got, want in [('headlines',5,5),('primary_texts',5,5),('ctas',3,3),('top_combinations',3,3)]:
    actual = len(copy_data.get(label,[]) if copy_data else [])
    print(f'  {label:<20}: {actual} / {want}  {"✅" if actual==want else "❌"}')

print()
print('CopyVariant schema validation (headlines):')
for i, h in enumerate(headlines):
    try:
        CopyVariant(**h)
        print(f'  headline #{i+1} ✅')
    except ValidationError as e:
        print(f'  headline #{i+1} ❌  {e.errors()[0]["msg"]}')

print()
print('Meta headline limit (≤40 chars):')
for h in headlines:
    t  = h.get('text','')
    ok = len(t) <= 40
    print(f'  {"✅" if ok else "⚠️ "} [{len(t):2d} chars] {t}')


Count checks (must match CopyResponse schema):
  headlines           : 5 / 5  ✅
  primary_texts       : 5 / 5  ✅
  ctas                : 3 / 3  ✅
  top_combinations    : 3 / 3  ✅

CopyVariant schema validation (headlines):
  headline #1 ✅
  headline #2 ✅
  headline #3 ✅
  headline #4 ✅
  headline #5 ✅

Meta headline limit (≤40 chars):
  ⚠️  [44 chars] Limited Time: 30% Off Premium Running Shoes!
  ✅ [39 chars] Don't Miss Out: Free Shipping Over €50!
  ⚠️  [41 chars] Hurry, Limited Stock of Discounted Shoes!
  ✅ [35 chars] Your Cart Awaits: Grab 30% Off Now!
  ✅ [40 chars] Exclusive 30% Discount on Running Shoes!


### 5b. Top combinations - what the frontend will display


In [ ]:
print('Top combinations (as shown in the CopyResponse frontend card):')
print('=' * 70)
for i, c in enumerate(combos, 1):
    print(f'\n#{i}  score={c.get("total_score")}  predicted CTR={c.get("predicted_ctr")}')
    print(f'Headline : {c.get("headline","")}')
    print(f'Body : {c.get("primary_text","")[:120]}')
    print(f'CTA : {c.get("cta","")}')


Top combinations (as shown in the CopyResponse frontend card):

#1  score=9.7  predicted CTR=9.1%-10.3%
Headline : Limited Time: 30% Off Premium Running Shoes!
Body : You've seen them, now get them! Our premium running shoes are 30% off for a limited time. Don't miss out!
CTA : Shop Now

#2  score=9.6  predicted CTR=8.9%-10.2%
Headline : Your Cart Awaits: Grab 30% Off Now!
Body : Get back to your cart and save big! 30% off on our premium running shoes ends soon. Free shipping over €50.
CTA : Claim Your Discount

#3  score=9.4  predicted CTR=8.8%-9.9%
Headline : Hurry, Limited Stock of Discounted Shoes!
Body : Act fast! Limited stock available for our top-rated running shoes at a 30% discount. Free shipping on orders over €50.
CTA : Grab The Deal


---
## 6. Consistency test - 3 calls, same Meta data

We call `analyse_campaigns()` three times with identical input and check
whether the **same problem campaigns** appear in all three responses.
If different campaigns are flagged each time, the prompt needs tightening.


In [ ]:
print('3x gpt-4o-mini with the same Meta payload...\n')
runs = []
for i in range(3):
    r = await call_llm(ANALYSIS_SYSTEM, meta_user_prompt, temperature=0.3)
    runs.append(r)
    titles = [rec.get('title','')[:45]
              for rec in (r['data'].get('recommendations',[]) if r['data'] else [])]
    print(f'Run #{i+1}  {r["latency_ms"]} ms  {r["tokens"]} tokens')
    for t in titles: print(f'  — {t}')
    print()

# Which campaigns are flagged consistently?
flagged_per_run = []
for r in runs:
    if not r['data']: continue
    flagged = set()
    for rec in r['data'].get('recommendations', []):
        body = rec.get('description','') + rec.get('action','')
        for name in real_campaign_names:
            if name[:20] in body: flagged.add(name)
    flagged_per_run.append(flagged)

if len(flagged_per_run) == 3:
    always = flagged_per_run[0].intersection(*flagged_per_run[1:])
    any_time = flagged_per_run[0].union(*flagged_per_run[1:])
    counts = [len(r['data'].get('recommendations',[])) for r in runs if r['data']]
    print('Flagged in ALL 3 runs  :', always  or 'none — prompt may be too generic')
    print('Flagged in ANY run :', any_time or 'none')
    print('Rec counts per run :', counts)
    print('Structure stable :', '✅' if max(counts)-min(counts) <= 1 else '⚠️  varies')


3x gpt-4o-mini with the same Meta payload...

Run #1  8343 ms  1895 tokens
  — Increase budget for high-performing retargeti
  — Pause underperforming prospecting campaign
  — Optimize creative for low engagement video ca
  — Reassess targeting for competitor conquest ca
  — Increase budget for high-converting cart aban

Run #2  7899 ms  1905 tokens
  — Optimize Black Friday - Prospecting
  — Increase Budget for Dynamic Product Ads - Car
  — Pause Brand Awareness - Video
  — Revise Spring Collection - Lookalike 3%
  — Enhance Competitor Conquest - Search Style

Run #3  8363 ms  1969 tokens
  — Increase Budget for High-Performing Retargeti
  — Pause Underperforming Spring Collection Campa
  — Optimize Dynamic Product Ads for Higher Conve
  — Revise Strategy for Brand Awareness Campaign
  — Adjust Bidding Strategy for Competitor Conque

Flagged in ALL 3 runs  : {'Dynamic Product Ads - Cart Abandoners', 'Competitor Conquest - Search Style', 'Spring Collection - Broad EU', 'Spring Collecti

---
## 7. Summary


In [ ]:
print('=' * 62)
print('  AI SERVICE EVAL SUMMARY')
print('=' * 62)

results = [
    ('analyse_campaigns (Meta)', meta_result, 'gpt-4o-mini'),
    ('analyse_campaigns (Google)', google_result, 'gpt-4o-mini'),
    ('generate_copy', copy_result, 'gpt-4o'),
]
for name, res, model in results:
    ok = res['error'] is None and res['data'] is not None
    print(f'\n  {name} ({model})')
    print(f'Status : {"✅ OK" if ok else "❌ FAILED"}')
    print(f'Latency : {res["latency_ms"]} ms')
    print(f'Tokens : {res["tokens"]}')
    if res['error']: print(f'Error : {res["error"]}')


  AI SERVICE EVAL SUMMARY

  analyse_campaigns (Meta) (gpt-4o-mini)
Status : ✅ OK
Latency : 10768 ms
Tokens : 2051

  analyse_campaigns (Google) (gpt-4o-mini)
Status : ✅ OK
Latency : 9961 ms
Tokens : 1799

  generate_copy (gpt-4o)
Status : ✅ OK
Latency : 8162 ms
Tokens : 1620
